<a href="https://colab.research.google.com/github/whgusdn5221/comfycolab/blob/main/sdxl_v1_0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# 1. 시스템 환경 최적화 (TCMalloc 및 Aria2 설치)
!apt -y update -qq
!wget https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4
!apt -y install -qq aria2

# 2. 파이썬 라이브러리 설치 (최신 규격으로 버전 오류 방지)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers triton mediapipe addict yapf fvcore omegaconf

# 3. ComfyUI 본체 및 필수 커스텀 노드 자동 설치
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# 커스텀 노드(매니저 및 IP-Adapter) 설치
!git clone https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/ComfyUI-Manager
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus

# [핵심] IP-Adapter 노드가 로딩되지 않는 문제를 해결하기 위한 필수 부품 설치
!pip install -q insightface onnxruntime-gpu

# 4. 접속 주소 생성 (SyntaxWarning 경고 해결 버전)
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64
import atexit, requests, subprocess, time, re, os
from random import randint
from threading import Timer
from queue import Queue

def cloudflared(port, metrics_port, output_queue):
    atexit.register(lambda p: p.terminate(), subprocess.Popen(['/content/cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
    attempts, tunnel_url = 0, None
    while attempts < 10 and not tunnel_url:
        attempts += 1
        time.sleep(3)
        try:
            # 문자열 앞에 r을 붙여 역슬래시 경고를 제거했습니다.
            tunnel_url = re.search(r"(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text).group("url")
        except:
            pass
    if not tunnel_url:
        raise Exception("Can't connect to Cloudflare Edge")
    output_queue.put(tunnel_url)

output_queue, metrics_port = Queue(), randint(8100, 9000)
thread = Timer(2, cloudflared, args=(8188, metrics_port, output_queue))
thread.start()
thread.join()
tunnel_url = output_queue.get()
os.environ['webui_url'] = tunnel_url
print(f"\n🚀 작업 준비 완료! 접속 주소: {tunnel_url}\n")

# 5. 작가님 전용 무기고 (모델 자동 다운로드)
# 실사 모델 1: XXMix_9 Realistic SDXL
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://civitai.com/api/download/models/112902" -d /content/ComfyUI/models/checkpoints -o xxmix9realisticsdxl_v10.safetensors
# 실사 모델 2: RealVisXL V5.0
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://civitai.com/api/download/models/302830" -d /content/ComfyUI/models/checkpoints -o realvisxlV50_v50LightningBakedvae.safetensors
# 만화 모델: Prefect Illustrious XL V7.0
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://civitai.com/api/download/models/1155099" -d /content/ComfyUI/models/checkpoints -o prefectIllustriousXL_v70.safet